In [9]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [10]:
import numpy as np
from datetime import datetime, timezone, timedelta
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import hdbscan
import uuid

## Step 1 — Embedding Model

Before running this, make sure **Settings → Internet** is turned ON (requires phone verification on Kaggle).

In [11]:
# =========================================================
# STEP 1 — EMBEDDING MODEL (load once, reused for every tweet)
# =========================================================
print("Loading embedding model...")
embedding_model = SentenceTransformer("l3cube-pune/indic-sentence-similarity-sbert")
# Alternative: SentenceTransformer("sentence-transformers/LaBSE")

def generate_embedding(text: str) -> np.ndarray:
    return embedding_model.encode(text, normalize_embeddings=True)

print("Model loaded successfully.")

Loading embedding model...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/7.73k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/676 [00:00<?, ?B/s]

... (9 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully.
Loading embedding model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Model loaded successfully.


## Step 2 — Cluster Data Structure

In [12]:
# =========================================================
# STEP 2 — CLUSTER DATA STRUCTURE
# =========================================================
class Cluster:
    def __init__(self, tweet_record: dict, embedding: np.ndarray):
        self.cluster_id = str(uuid.uuid4())
        self.members = [tweet_record]
        self.embeddings = [embedding]
        self.centroid = embedding.copy()
        self.created_at = tweet_record["created_at"]
        self.last_updated_at = tweet_record["created_at"]
        self.dept_relevance = dict(tweet_record["needs"])

    def add_member(self, tweet_record: dict, embedding: np.ndarray):
        self.members.append(tweet_record)
        self.embeddings.append(embedding)
        self.centroid = np.mean(self.embeddings, axis=0)
        # keep last_updated_at as the MOST RECENT timestamp seen so far
        if tweet_record["created_at"] > self.last_updated_at:
            self.last_updated_at = tweet_record["created_at"]
        for dept, score in tweet_record["needs"].items():
            self.dept_relevance[dept] = max(self.dept_relevance.get(dept, 0), score)

    def size(self) -> int:
        return len(self.members)

    def summary(self) -> str:
        texts = [m["text"][:50] for m in self.members]
        return f"Cluster[{self.cluster_id[:8]}] size={self.size()} dept_relevance={self.dept_relevance} members={texts}"

## Step 3 — Cluster Manager (Incremental Assignment + Periodic HDBSCAN Correction)

In [13]:
# =========================================================
# STEP 3 — CLUSTER MANAGER (incremental assignment + periodic correction)
# =========================================================
class ClusterManager:
    def __init__(self, similarity_threshold=0.32, time_window_hours=48, min_cluster_size=3,
                 dept_overlap_threshold=0.3):
        self.clusters = []
        self.similarity_threshold = similarity_threshold
        self.time_window_hours = time_window_hours
        self.min_cluster_size = min_cluster_size
        self.dept_overlap_threshold = dept_overlap_threshold
        self.tweets_since_last_recluster = 0

    def _time_diff_hours(self, t1: str, t2: str) -> float:
        d1 = datetime.fromisoformat(t1)
        d2 = datetime.fromisoformat(t2)
        return abs((d1 - d2).total_seconds()) / 3600

    def _dept_overlap(self, cluster: Cluster, tweet_needs: dict) -> bool:
        """
        Soft guard — blocks merges only when NO department is jointly relevant.

        FIX (was too strict): originally required BOTH the tweet AND the cluster
        to independently cross 0.5 on the SAME department. This wrongly blocked
        legitimate cross-department events -- e.g. a flood tweet describing its
        HEALTH consequence (healthcare=0.90, municipal=0.40) failing to merge
        into a cluster whose aggregated relevance is municipal-dominant
        (municipal=0.99, healthcare=0.05) -- neither department crossed 0.5 on
        BOTH sides, even though the tweet is clearly part of the same real event.

        Relaxed version: lower the bar to dept_overlap_threshold (default 0.3)
        on both sides. This still blocks genuinely unrelated departments (e.g.
        flood vs. robbery, where every cross-department score stays near 0),
        while allowing a real multi-department event (flood -> health impact)
        to merge correctly.
        """
        for dept, score in tweet_needs.items():
            if score > self.dept_overlap_threshold and \
               cluster.dept_relevance.get(dept, 0) > self.dept_overlap_threshold:
                return True
        return False

    def assign_tweet(self, tweet_record: dict):
        """
        MECHANISM 1 — Incremental Assignment.
        Runs on EVERY new tweet. Fast: compares only against active clusters.
        """
        text = tweet_record["text"]
        embedding = generate_embedding(text)

        best_cluster = None
        best_similarity = -1.0

        for cluster in self.clusters:
            sim = cosine_similarity([embedding], [cluster.centroid])[0][0]

            if sim < self.similarity_threshold:
                continue

            time_gap = self._time_diff_hours(tweet_record["created_at"], cluster.last_updated_at)
            if time_gap > self.time_window_hours:
                continue

            if not self._dept_overlap(cluster, tweet_record["needs"]):
                continue

            if sim > best_similarity:
                best_similarity = sim
                best_cluster = cluster

        if best_cluster is not None:
            best_cluster.add_member(tweet_record, embedding)
            result = (best_cluster, "joined_existing", round(float(best_similarity), 4))
        else:
            new_cluster = Cluster(tweet_record, embedding)
            self.clusters.append(new_cluster)
            result = (new_cluster, "new_cluster", None)

        self.tweets_since_last_recluster += 1
        return result

    def periodic_recluster(self):
        """
        MECHANISM 2 — Periodic HDBSCAN Correction.
        Runs OCCASIONALLY (call manually every N tweets / N minutes).
        Operates on cluster CENTROIDS, not raw tweets — stays fast.
        """
        if len(self.clusters) < 2:
            self.tweets_since_last_recluster = 0
            return 0  # nothing to correct

        all_centroids = np.array([c.centroid for c in self.clusters])

        clusterer = hdbscan.HDBSCAN(min_cluster_size=2, metric="euclidean")
        labels = clusterer.fit_predict(all_centroids)

        groups = {}
        for idx, label in enumerate(labels):
            if label == -1:
                continue  # noise — HDBSCAN says leave this cluster alone
            groups.setdefault(label, []).append(idx)

        merges_done = 0
        indices_to_remove = set()
        new_clusters = []

        for label, idx_list in groups.items():
            if len(idx_list) < 2:
                continue
            base = self.clusters[idx_list[0]]
            for other_idx in idx_list[1:]:
                other = self.clusters[other_idx]
                for member, emb in zip(other.members, other.embeddings):
                    base.add_member(member, emb)
                indices_to_remove.add(other_idx)
                merges_done += 1
            new_clusters.append(base)

        # rebuild cluster list: keep untouched clusters + merged bases
        final_list = []
        seen_ids = set()
        for idx, c in enumerate(self.clusters):
            if idx in indices_to_remove:
                continue
            if c.cluster_id not in seen_ids:
                final_list.append(c)
                seen_ids.add(c.cluster_id)

        self.clusters = final_list
        self.tweets_since_last_recluster = 0
        return merges_done

    def get_situations(self):
        """Clusters with enough corroboration become 'situations' ready for Priority Scoring."""
        situations = [c for c in self.clusters if c.size() >= self.min_cluster_size]
        holding_pool = [c for c in self.clusters if c.size() < self.min_cluster_size]
        return situations, holding_pool

    def print_state(self):
        print(f"\n--- Cluster Manager State: {len(self.clusters)} active cluster(s) ---")
        for c in self.clusters:
            print(f"  {c.summary()}")

## Helper — build a fake combined-record (matches your Combine step's output)

In [14]:
def make_tweet(tweet_id, text, hours_offset, needs, severity=0.5, actionability=0.5, base_time=None):
    if base_time is None:
        base_time = datetime(2026, 9, 2, 0, 0, 0, tzinfo=timezone.utc)
    ts = (base_time + timedelta(hours=hours_offset)).isoformat()
    return {
        "tweet_id": tweet_id,
        "text": text,
        "created_at": ts,
        "needs": needs,
        "severity": severity,
        "actionability": actionability,
    }

## Step 4 — Threshold Calibration

**What we learned from the first real run:** the calibration numbers below (0.31–0.62 for same-event pairs, ~0.08 for different-event pairs) confirmed the embedding model itself works well — but Test 1 still failed even though t1-vs-t3 similarity was 0.6213, well above the threshold used. That proved the bug wasn't the similarity threshold at all — it was the **department-overlap guard**, which originally required BOTH the tweet and the cluster to independently cross 0.5 on the SAME department. A flood tweet describing its health impact (healthcare=0.90, municipal=0.40) couldn't satisfy that against a municipal-dominant cluster (municipal=0.99, healthcare=0.05) on either department, so it was blocked regardless of how similar the text was.

**Two fixes applied (see Step 3 above):**
1. `_dept_overlap()` threshold relaxed from requiring both sides >0.5 to both sides >0.3 (`dept_overlap_threshold`) — still blocks genuinely unrelated departments (flood vs. robbery stay near 0 overlap), but allows a real cross-department event like flood→health-impact to merge correctly.
2. `similarity_threshold` recalibrated from 0.45 down to **0.32** — just above Group B's observed 0.08 average with a safety margin, and just below Group A's observed 0.31 minimum, based on the actual printed numbers below (not a guess).

In [15]:
# =========================================================
# THRESHOLD CALIBRATION — run BEFORE the test suite
# =========================================================
same_event_texts = [
    "flood in my area need help urgently, road submerged",
    "water entering homes, whole street flooded",
    "people getting sick from the flood water, need medical help",
    "cars stuck in flooded street, road completely submerged",
]

different_event_texts = [
    "flood in my area need help urgently, road submerged",
    "armed robbery happening near the market, send police",
]

print("=" * 70)
print("GROUP A — same real-world event (SHOULD merge, expect HIGHER similarity)")
print("=" * 70)
same_event_scores = []
for i in range(len(same_event_texts)):
    for j in range(i + 1, len(same_event_texts)):
        emb_i = generate_embedding(same_event_texts[i])
        emb_j = generate_embedding(same_event_texts[j])
        sim = cosine_similarity([emb_i], [emb_j])[0][0]
        same_event_scores.append(sim)
        print(f"  [{i}] vs [{j}]: {sim:.4f}")
print(f"  Average: {np.mean(same_event_scores):.4f}  |  Min: {np.min(same_event_scores):.4f}")

print("\n" + "=" * 70)
print("GROUP B — different events (should NOT merge, expect LOWER similarity)")
print("=" * 70)
diff_event_scores = []
for i in range(len(different_event_texts)):
    for j in range(i + 1, len(different_event_texts)):
        emb_i = generate_embedding(different_event_texts[i])
        emb_j = generate_embedding(different_event_texts[j])
        sim = cosine_similarity([emb_i], [emb_j])[0][0]
        diff_event_scores.append(sim)
        print(f"  [{i}] vs [{j}]: {sim:.4f}")
print(f"  Average: {np.mean(diff_event_scores):.4f}")

print("\n" + "=" * 70)
suggested_threshold = (np.min(same_event_scores) + np.max(diff_event_scores)) / 2 if diff_event_scores else np.min(same_event_scores) - 0.05
print(f"SUGGESTED THRESHOLD (midpoint between groups): {suggested_threshold:.4f}")
print("=" * 70)
print("If GROUP A's minimum score is comfortably ABOVE GROUP B's maximum score,")
print("any threshold between them works. Use the suggested value below, or adjust")
print("similarity_threshold in the test cells to match your own printed numbers.")

GROUP A — same real-world event (SHOULD merge, expect HIGHER similarity)
  [0] vs [1]: 0.4935
  [0] vs [2]: 0.6213
  [0] vs [3]: 0.5525
  [1] vs [2]: 0.4968
  [1] vs [3]: 0.5826
  [2] vs [3]: 0.3094
  Average: 0.5094  |  Min: 0.3094

GROUP B — different events (should NOT merge, expect LOWER similarity)
  [0] vs [1]: 0.0793
  Average: 0.0793

SUGGESTED THRESHOLD (midpoint between groups): 0.1943
If GROUP A's minimum score is comfortably ABOVE GROUP B's maximum score,
any threshold between them works. Use the suggested value below, or adjust
similarity_threshold in the test cells to match your own printed numbers.
GROUP A — same real-world event (SHOULD merge, expect HIGHER similarity)
  [0] vs [1]: 0.4935
  [0] vs [2]: 0.6213
  [0] vs [3]: 0.5525
  [1] vs [2]: 0.4968
  [1] vs [3]: 0.5826
  [2] vs [3]: 0.3094
  Average: 0.5094  |  Min: 0.3094

GROUP B — different events (should NOT merge, expect LOWER similarity)
  [0] vs [1]: 0.0793
  Average: 0.0793

SUGGESTED THRESHOLD (midpoint betw

## Step 5 — Test Suite

Thresholds below use **similarity_threshold=0.32** and **dept_overlap_threshold=0.3** — both recalibrated from the actual numbers printed in Step 4, and both fixes (guard relaxation + threshold) are needed together, since the calibration run showed the department-overlap guard, not similarity alone, was blocking the legitimate cross-department merge in Test 1.

In [16]:
# =========================================================
# TEST SUITE — multiple scenarios
# =========================================================

def check(condition, pass_msg, fail_msg):
    if condition:
        print(f"  PASS: {pass_msg}")
    else:
        print(f"  FAIL: {fail_msg}")

print("\n" + "=" * 70)
print("TEST 1 — Same event, different tweets, should cluster together")
print("=" * 70)
manager1 = ClusterManager(similarity_threshold=0.32, time_window_hours=48, min_cluster_size=3,
                           dept_overlap_threshold=0.3)
test1_tweets = [
    make_tweet("t1", "flood in my area need help urgently, road submerged", 0.0,
               {"healthcare": 0.01, "municipal": 0.99, "tourism": 0.01, "law": 0.01}),
    make_tweet("t2", "water entering homes, whole street flooded", 0.5,
               {"healthcare": 0.05, "municipal": 0.95, "tourism": 0.01, "law": 0.01}),
    make_tweet("t3", "people getting sick from the flood water, need medical help", 1.0,
               {"healthcare": 0.90, "municipal": 0.40, "tourism": 0.01, "law": 0.01}),
    make_tweet("t4", "cars stuck in flooded street, road completely submerged", 1.5,
               {"healthcare": 0.02, "municipal": 0.97, "tourism": 0.01, "law": 0.01}),
]
for tw in test1_tweets:
    cluster, action, sim = manager1.assign_tweet(tw)
    print(f"  {tw['tweet_id']}: {action} (sim={sim}) -> cluster {cluster.cluster_id[:8]}")
manager1.print_state()
situations, holding = manager1.get_situations()
print(f"  => {len(situations)} situation(s), {len(holding)} in holding pool")
check(len(manager1.clusters) == 1,
      "all flood-related tweets merged into a single cluster (incl. the cross-department health-impact tweet)",
      f"expected 1 cluster, got {len(manager1.clusters)}")


print("\n" + "=" * 70)
print("TEST 2 — Two unrelated events, should form separate clusters")
print("=" * 70)
manager2 = ClusterManager(similarity_threshold=0.32, time_window_hours=48, min_cluster_size=2,
                           dept_overlap_threshold=0.3)
test2_tweets = [
    make_tweet("t1", "flood in my area need help urgently, road submerged", 0.0,
               {"healthcare": 0.01, "municipal": 0.99, "tourism": 0.01, "law": 0.01}),
    make_tweet("t2", "water entering homes, whole street flooded", 0.5,
               {"healthcare": 0.05, "municipal": 0.95, "tourism": 0.01, "law": 0.01}),
    make_tweet("t3", "armed robbery happening near the market, send police", 0.7,
               {"healthcare": 0.01, "municipal": 0.02, "tourism": 0.01, "law": 0.97}),
    make_tweet("t4", "someone is being robbed at gunpoint on main street", 1.0,
               {"healthcare": 0.01, "municipal": 0.03, "tourism": 0.01, "law": 0.95}),
]
for tw in test2_tweets:
    cluster, action, sim = manager2.assign_tweet(tw)
    print(f"  {tw['tweet_id']}: {action} (sim={sim}) -> cluster {cluster.cluster_id[:8]}")
manager2.print_state()
check(len(manager2.clusters) == 2,
      "flood and robbery events correctly stayed in separate clusters",
      f"expected 2 clusters, got {len(manager2.clusters)} — dept_overlap_threshold or similarity_threshold too loose")


print("\n" + "=" * 70)
print("TEST 3 — Same wording, different time window (>48h apart), should NOT merge")
print("=" * 70)
manager3 = ClusterManager(similarity_threshold=0.32, time_window_hours=48, min_cluster_size=2,
                           dept_overlap_threshold=0.3)
test3_tweets = [
    make_tweet("t1", "flood in my area need help urgently, road submerged", 0.0,
               {"healthcare": 0.01, "municipal": 0.99, "tourism": 0.01, "law": 0.01}),
    make_tweet("t2", "flood in my area need help urgently, road submerged again", 100.0,  # +100 hours later
               {"healthcare": 0.01, "municipal": 0.99, "tourism": 0.01, "law": 0.01}),
]
for tw in test3_tweets:
    cluster, action, sim = manager3.assign_tweet(tw)
    print(f"  {tw['tweet_id']}: {action} (sim={sim}) -> cluster {cluster.cluster_id[:8]}")
manager3.print_state()
check(len(manager3.clusters) == 2,
      "time-window guard correctly prevented a stale merge",
      f"expected 2 clusters, got {len(manager3.clusters)} — time_window_hours needs adjustment")


print("\n" + "=" * 70)
print("TEST 4 — Cross-lingual: English + Tamil about same event, should merge")
print("=" * 70)
manager4 = ClusterManager(similarity_threshold=0.32, time_window_hours=48, min_cluster_size=2,
                           dept_overlap_threshold=0.3)
test4_tweets = [
    make_tweet("t1", "flood in Chennai, roads are submerged, need urgent help", 0.0,
               {"healthcare": 0.02, "municipal": 0.98, "tourism": 0.01, "law": 0.01}),
    make_tweet("t2", "சென்னையில் வெள்ளம், சாலைகள் மூழ்கியுள்ளன, உதவி தேவை", 0.3,  # Tamil: same meaning
               {"healthcare": 0.03, "municipal": 0.96, "tourism": 0.01, "law": 0.01}),
]
for tw in test4_tweets:
    cluster, action, sim = manager4.assign_tweet(tw)
    print(f"  {tw['tweet_id']}: {action} (sim={sim}) -> cluster {cluster.cluster_id[:8]}")
manager4.print_state()
check(len(manager4.clusters) == 1,
      "English and Tamil tweets about the same flood correctly merged (cross-lingual embedding working)",
      f"expected 1 cluster, got {len(manager4.clusters)}")


print("\n" + "=" * 70)
print("TEST 5 — Minimum cluster size gate (holding pool vs. situation)")
print("=" * 70)
manager5 = ClusterManager(similarity_threshold=0.32, time_window_hours=48, min_cluster_size=3,
                           dept_overlap_threshold=0.3)
test5_tweets = [
    make_tweet("t1", "flood in my area, road submerged", 0.0,
               {"healthcare": 0.01, "municipal": 0.99, "tourism": 0.01, "law": 0.01}),
    make_tweet("t2", "isolated report of a minor pothole issue", 0.2,
               {"healthcare": 0.01, "municipal": 0.60, "tourism": 0.01, "law": 0.01}),
]
for tw in test5_tweets:
    cluster, action, sim = manager5.assign_tweet(tw)
situations, holding = manager5.get_situations()
print(f"  situations (>=3 members): {len(situations)}")
print(f"  holding pool (<3 members): {len(holding)}")
check(len(situations) == 0 and len(holding) == 2,
      "single/small clusters correctly held back from becoming 'situations'",
      f"expected 0 situations / 2 holding, got {len(situations)} situations / {len(holding)} holding")


print("\n" + "=" * 70)
print("TEST 6 — Periodic HDBSCAN correction merges drifted clusters")
print("=" * 70)
manager6 = ClusterManager(similarity_threshold=0.85, time_window_hours=48, min_cluster_size=2,
                           dept_overlap_threshold=0.3)  # deliberately high sim threshold -> forces drift
test6_tweets = [
    make_tweet("t1", "flood in my area, road submerged", 0.0,
               {"healthcare": 0.01, "municipal": 0.99, "tourism": 0.01, "law": 0.01}),
    make_tweet("t2", "water flooding the street near my house", 0.5,
               {"healthcare": 0.01, "municipal": 0.97, "tourism": 0.01, "law": 0.01}),
]
for tw in test6_tweets:
    cluster, action, sim = manager6.assign_tweet(tw)
    print(f"  {tw['tweet_id']}: {action} (sim={sim})")
print(f"  Before correction: {len(manager6.clusters)} cluster(s)")
merges = manager6.periodic_recluster()
print(f"  HDBSCAN correction merged {merges} pair(s)")
print(f"  After correction: {len(manager6.clusters)} cluster(s)")
check(len(manager6.clusters) <= 2,
      "HDBSCAN correction pass ran without error (merge outcome depends on actual embedding distances)",
      "unexpected cluster count after correction")


print("\n" + "=" * 70)
print("ALL TESTS COMPLETE")
print("=" * 70)


TEST 1 — Same event, different tweets, should cluster together
  t1: new_cluster (sim=None) -> cluster 77a62a07
  t2: joined_existing (sim=0.4935) -> cluster 77a62a07
  t3: joined_existing (sim=0.647) -> cluster 77a62a07
  t4: joined_existing (sim=0.579) -> cluster 77a62a07

--- Cluster Manager State: 1 active cluster(s) ---
  Cluster[77a62a07] size=4 dept_relevance={'healthcare': 0.9, 'municipal': 0.99, 'tourism': 0.01, 'law': 0.01} members=['flood in my area need help urgently, road submerge', 'water entering homes, whole street flooded', 'people getting sick from the flood water, need med', 'cars stuck in flooded street, road completely subm']
  => 1 situation(s), 0 in holding pool
  PASS: all flood-related tweets merged into a single cluster (incl. the cross-department health-impact tweet)

TEST 2 — Two unrelated events, should form separate clusters
  t1: new_cluster (sim=None) -> cluster c3d1ff18
  t2: joined_existing (sim=0.4935) -> cluster c3d1ff18
  t3: new_cluster (sim=None)